In [ ]:
import cv2
import numpy as np
import torch
import os
import pandas as pd
from tqdm import tqdm

# Setup paths
csv_path = "/msrvtt_train_2k_fullcaptions.csv"
output_dir = "/processed_frames"
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path)
processed_paths = []

def extract_and_save(video_path, video_id, num_frames=8):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames < num_frames:
        cap.release()
        return None # Skip broken/too short videos

    # Uniform sampling logic
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    
    current_idx = 0
    while len(frames) < num_frames:
        ret, frame = cap.read()
        if not ret: break
        if current_idx in indices:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)
        current_idx += 1
    cap.release()

    if len(frames) == num_frames:
        # Stack to (8, 224, 224, 3) -> Convert to (8, 3, 224, 224)
        video_np = np.stack(frames).transpose(0, 3, 1, 2)
        save_path = os.path.join(output_dir, f"{video_id}.npz")
        np.savez_compressed(save_path, video=video_np)
        return save_path
    return None

# Process all videos
for idx, row in tqdm(df.iterrows(), total=len(df)):
    save_path = extract_and_save(row['video_path'], row['video_id'])
    processed_paths.append(save_path)

# Update CSV with the new paths to the saved NUMPY files
df['processed_path'] = processed_paths
df.dropna(subset=['processed_path'], inplace=True) # Remove failures
df.to_csv("msrvtt_2k_preprocessed.csv", index=False)

100%|██████████| 40000/40000 [42:31<00:00, 15.67it/s]   
